# Ingeniero de boxes para simracing

**Carlos Peñalver · Proyecto del módulo de IA Generativa**

Gemini, Chroma y LangGraph. Chat con memoria y cinco casos reales documentados; evaluación técnica parcial.


## 0. Configuración

Usar el kernel de `.venv` de Proyecto. Configurar las credenciales en `Proyecto/.env`.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os
import sys

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from retrieval import GeminiRetrievalEmbeddings, index_documents
from agent import crear_grafo

load_dotenv(ROOT / ".env", override=True)
CORPUS_DIR = ROOT / "data/corpus"
CHROMA_DIR = ROOT / "data/chroma"
SYSTEM_PROMPT = (ROOT / "prompts/system.md").read_text(encoding="utf-8")


## 1. Corpus

75 fichas: LMU (Ultimate Setup Hub), iRacing (Chris Haye) y ACC (Driver61).
Cada ficha conserva el caso completo y sus metadatos. Detalle en `docs/FUENTES.md`.


In [ ]:
records = [json.loads(line) for line in
           (CORPUS_DIR / "mvp/preparado/corpus.jsonl").read_text(encoding="utf-8").splitlines()]
documents = [
    Document(page_content=record["text"], metadata={
        key: value for key, value in record.items() if key != "text"
    })
    for record in records
]


## 2. Embeddings y Chroma

Se reutiliza el índice persistente de las 75 fichas. La recuperación se ejecuta dentro del grafo, al enviar la pregunta de la sección 4.


### 2.1. Embeddings de Gemini

`GeminiRetrievalEmbeddings` adapta el cliente de clase a Embedding 2. Detalle en `retrieval.py`.


In [ ]:
MODEL_EMBEDDING = os.environ["GEMINI_EMBEDDING_MODEL"]
llm_embedding = GeminiRetrievalEmbeddings(
    model=MODEL_EMBEDDING,
    api_key=os.environ["GOOGLE_API_KEY"],
    vertexai=False,
    request_options={"timeout": 30000, "retry_options": {"attempts": 1}},
)


### 2.2. Índice persistente

Se guarda en `data/chroma/`. El mismo corpus reutiliza los embeddings existentes.


In [ ]:
vectorstore = index_documents(documents, llm_embedding, CHROMA_DIR)


## 3. System prompt

Se carga de `prompts/system.md`: voz de ingeniero de pista, recomendaciones directas, hipótesis delimitadas, citas `[ID]` y un cambio por prueba. Las fuentes se integran en el razonamiento sin repetir «según la guía». Justificación en el README.


## 4. Agente en LangGraph

`START → preparar → recuperar → responder → END`

El núcleo está en `agent.py`, siguiendo el patrón de estado, nodos y conexiones de clase. Antes de buscar revisa el contexto y añade el historial de la conversación. Recupera tres fichas del simulador seleccionado y envía los textos y sus metadatos a Gemini. Al cambiar de coche o simulador pide confirmación antes de continuar.

Cada consulta al agente consume un embedding de consulta y una generación, salvo cuando pide confirmar un contexto. Modelo en `.env`: `gemini-3.1-flash-lite`. Leer los ejemplos guardados no consume API.

Para ir al chat, basta con ejecutar la configuración, el corpus, el índice y la celda de esta sección que crea `app`. Las consultas de ejemplo son opcionales.


In [ ]:
MODEL = os.environ["GEMINI_MODEL"]
llm = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=os.environ["GOOGLE_API_KEY"],
    vertexai=False,
    temperature=1.0,
    max_output_tokens=1800,
    max_retries=1,
    timeout=45,
)
checkpointer = InMemorySaver()
app = crear_grafo(vectorstore, llm, SYSTEM_PROMPT, checkpointer=checkpointer)
config_conversacion = {"configurable": {"thread_id": uuid4().hex}}


**Consulta de ejemplo opcional** (consume API si se ejecuta en una celda de código):

```python
simulator = "lmu"  # lmu, iracing o acc
query = "En LMU, tengo sobreviraje al acelerar a la salida de una curva lenta. ¿Qué propone la guía?"

resultado = app.invoke(
    {"pregunta": query, "simulador": simulator},
    config=config_conversacion,
)
# El notebook muestra las citas para revisar; la web usará respuesta_piloto.
display(Markdown(resultado["respuesta"]))

if resultado["fuentes"]:
    display(Markdown("**Fuentes citadas**\n\n" + "\n".join(
        f"- [{fuente['id']}]({fuente['url']}) — {fuente['locator']}"
        for fuente in resultado["fuentes"]
    )))
```


### Respuesta real guardada: voz de ingeniero

Consulta LMU de la celda anterior. Revisión del 24/10/2026 con Gemini 3.1 Flash-Lite: una generación real sobre las mismas tres fichas recuperadas antes, sin repetir el embedding de consulta. No es una nueva búsqueda ni otro caso independiente. Las celdas están sin ejecutar para este prompt; al ejecutarlas se hará una consulta nueva.

Entendido. Si el sobreviraje aparece específicamente al aplicar potencia a la salida de una curva lenta, el problema suele estar en una transferencia de pesos hacia el eje trasero que no es lo suficientemente rápida o controlada [LMU_USH_24].

Empezaría por **reducir la rigidez de la barra estabilizadora trasera**. Esto debería ayudar a que la parte trasera del coche se asiente mejor al acelerar, mejorando la tracción longitudinal [LMU_USH_24].

Ten en cuenta lo siguiente:
*   Realiza el cambio en incrementos pequeños y prueba solo este ajuste para poder evaluar correctamente el resultado.
*   Si tras el cambio notas que el coche mejora su comportamiento, consérvalo. Si el sobreviraje persiste o el coche se vuelve impreciso, revierte el ajuste al valor anterior.
*   Esta recomendación asume que tu trazada es correcta; si notas que el sobreviraje empieza antes de pisar el acelerador, el diagnóstico podría cambiar [LMU_USH_24].

Si este ajuste no es suficiente, otra opción que sugiere la documentación es reducir el bloqueo de potencia del diferencial (*differential power locking*), lo cual ayuda a mantener la rotación bajo control durante la aceleración [LMU_USH_24].

¿El setup del coche te permite realizar ajustes en la barra estabilizadora trasera?

Fuentes:
[LMU_USH_24] https://ultimatesetuphub.com/race-engineer/n5n2n0

**Localizadores verificados**

- [LMU_USH_24](https://ultimatesetuphub.com/race-engineer/n5n2n0) — Ruta del árbol público / Oversteer / Corner Exit / Low Speed


## 5. Memoria por conversación

Conservar `config_conversacion` permite seguir la sesión anterior. El historial distingue los mensajes del piloto de las propuestas del ingeniero. Cada seguimiento vuelve a recuperar documentos; las respuestas anteriores no se usan como fuentes.

La memoria vive en RAM: se pierde al reiniciar el kernel o recrear `app`. Para una conversación independiente, usa otro identificador:

```python
config_conversacion = {"configurable": {"thread_id": uuid4().hex}}
```

El coche puede indicarse con `"coche": "nombre del coche"` en la llamada. Si se omite, se conserva el del contexto actual; al cambiar de simulador no se hereda el coche anterior. Completar un coche antes desconocido no borra la sesión.

Si cambias los campos de coche o simulador, la respuesta pide confirmación y no llama a Gemini. Para confirmar y procesar la pregunta que quedó pendiente:

```python
resultado = app.invoke({"confirmar_contexto": True}, config=config_conversacion)
display(Markdown(resultado["respuesta"]))
```

Al confirmar se inicia el contexto nuevo sin el historial de reglajes anterior. Si el cambio solo se menciona en el texto, el ingeniero debe pedir que se actualicen y confirmen los campos; no se cambia el filtro automáticamente.

Los dos ejemplos siguientes son seguimientos ficticios opcionales. Si los copias a una celda de código, cada uno consume una búsqueda y una generación.


**Consulta de ejemplo opcional** (consume API si se ejecuta en una celda de código):

```python
seguimiento = "Todavía no he cambiado nada. ¿Qué debería observar si pruebo solo tu primera propuesta?"

resultado = app.invoke({"pregunta": seguimiento}, config=config_conversacion)
display(Markdown(resultado["respuesta"]))
for fuente in resultado["fuentes"]:
    display(Markdown(f"[{fuente['id']}]({fuente['url']}) — {fuente['locator']}"))
```


**Consulta de ejemplo opcional** (consume API si se ejecuta en una celda de código):

```python
seguimiento = "He aplicado únicamente el primer ajuste que me propusiste; el resto sigue igual. Ahora noto algo menos de sobreviraje al acelerar, pero todavía aparece al salir de curvas lentas. ¿Cómo seguimos?"

resultado = app.invoke({"pregunta": seguimiento}, config=config_conversacion)
display(Markdown(resultado["respuesta"]))
for fuente in resultado["fuentes"]:
    display(Markdown(f"[{fuente['id']}]({fuente['url']}) — {fuente['locator']}"))
```


### Seguimientos reales guardados: escenario ficticio

La primera respuesta se recuperó del ejemplo de voz de la sección 4; no se volvió a generar. Se cargó ese intercambio real en una conversación temporal y se ejecutaron dos seguimientos con recuperación y generación nuevas. Son sensaciones ficticias del piloto, no pruebas en pista. Las celdas de código están sin ejecutar; este bloque conserva las salidas reales para poder leerlas sin API.

**Piloto, seguimiento 1:** Todavía no he cambiado nada. ¿Qué debería observar si pruebo solo tu primera propuesta?

**Ingeniero:**

Para confirmar el contexto: todavía no has especificado qué coche estás pilotando en Le Mans Ultimate. Por favor, confírmalo antes de realizar cambios, ya que los ajustes disponibles pueden variar significativamente según el modelo.

Sobre tu pregunta, al reducir la rigidez de la barra estabilizadora trasera, buscas suavizar la suspensión trasera para permitir que la transferencia de pesos hacia el eje posterior ocurra de manera más rápida y efectiva al pisar el acelerador [LMU_USH_24].

Al probar este ajuste, deberías observar lo siguiente:

* **Comportamiento en tracción:** El coche debería sentirse más asentado al aplicar gas, reduciendo esa tendencia a deslizar de atrás que sientes ahora. Si notas que la parte trasera se vuelve más predecible y sientes que puedes aplicar potencia antes o con mayor confianza sin que el coche quiera sobrevirar, el ajuste está funcionando.
* **Respuesta del chasis:** Fíjate en si el coche se siente "más pesado" o menos ágil al iniciar el giro; un ajuste excesivo podría hacer que el coche se sienta lento o subvirador al entrar en la curva.
* **Cuándo revertir:** Si tras el cambio notas que el coche ha perdido demasiada agilidad en la entrada o si, por el contrario, no notas ninguna mejora en la tracción al salir, revierte el ajuste al valor original.

Si el sobreviraje persiste, esto apunta a que la transferencia de pesos sigue siendo insuficiente o demasiado lenta para la cantidad de par que estás aplicando [LMU_USH_24]. 

¿Qué coche estás usando para poder validar si este ajuste está disponible en tu setup?

- [LMU_USH_24](https://ultimatesetuphub.com/race-engineer/n5n2n0) — Ruta del árbol público / Oversteer / Corner Exit / Low Speed

**Piloto, seguimiento 2:** He reducido la rigidez de la barra estabilizadora trasera que propusiste. Ahora noto algo menos de sobreviraje al acelerar, pero todavía aparece al salir de curvas lentas. ¿Cómo seguimos?

**Ingeniero:**

Comprendido. Has confirmado que el ajuste de la barra estabilizadora trasera ha dado una mejora parcial, lo que valida la dirección de trabajo: facilitar la transferencia de pesos hacia el eje trasero [LMU_USH_24].

Como el sobreviraje persiste al acelerar en curvas lentas, el siguiente paso lógico es seguir ablandando la suspensión trasera para que esa transición de pesos sea aún más efectiva. Te recomiendo **reducir la rigidez de los muelles traseros** [LMU_USH_24].

Al igual que con la barra estabilizadora, aplica este cambio en incrementos pequeños:

* **Qué observar:** Este ajuste busca que el chasis se comprima más fácilmente sobre el eje trasero al aplicar potencia, mejorando el agarre mecánico. Deberías notar que el coche tiene un poco más de tracción al salir de la curva y que la trasera se siente más asentada al pisar el acelerador.
* **Cuándo revertir:** Si notas que el coche empieza a tocar fondo (*bottoming out*) o se vuelve demasiado errático en baches o cambios de dirección rápidos, revierte este ajuste a su valor anterior, ya que habrás sobrepasado el límite de compromiso de la suspensión.

Si después de esto el sobreviraje sigue siendo un problema, otra vía documentada es **reducir los *clickers* de compresión lenta (slow bump) trasera** [LMU_USH_24]. Esto permitiría que el amortiguador ceda más rápido al aplicar gas, complementando el efecto del muelle.

Por favor, confirma qué coche utilizas cuando puedas, ya que es necesario para verificar si estos ajustes están disponibles en tu menú de configuración. ¿Qué modelo llevas en este LMU?

- [LMU_USH_24](https://ultimatesetuphub.com/race-engineer/n5n2n0) — Ruta del árbol público / Oversteer / Corner Exit / Low Speed


## 6. Chat interactivo y casos documentados

Ejecuta la celda de funciones y después `iniciar_chat`. El chat empieza una conversación nueva de LMU, separada de los ejemplos. Cada mensaje normal consume una búsqueda y una generación. Las confirmaciones de cambio se solicitan antes de gastar API.

Comandos: `/salir`, `/nuevo`, `/simulador acc` (también `lmu` o `iracing`), `/coche nombre`, `/confirmar` y `/cancelar`. Tras seleccionar otro coche o simulador, escribe la consulta y confirma cuando se solicite. `/nuevo` empieza otra conversación con la selección actual.

Las citas siguen visibles aquí para evaluación; la futura web mostrará la respuesta limpia. La salida del chat debe revisarse antes de guardar o compartir el notebook si se han introducido datos privados.


In [ ]:
def iniciar_chat(app, simulador="lmu", coche=None, ver_fuentes=True, leer=None, escribir=None):
    """Chat del notebook; las entradas y salidas se pueden sustituir en pruebas locales."""
    leer = input if leer is None else leer
    escribir = (lambda texto: display(Markdown(texto))) if escribir is None else escribir
    permitidos = {"lmu", "iracing", "acc"}
    if simulador not in permitidos:
        raise ValueError("Selecciona lmu, iracing o acc.")
    seleccion = {"simulador": simulador, "coche": coche}
    config_chat = {"configurable": {"thread_id": uuid4().hex}}
    pendiente = False
    escribir(f"**Chat de boxes · {simulador.upper()}**. Escribe tu consulta o `/salir`. "
             "Comandos: `/nuevo`, `/simulador lmu|iracing|acc`, `/coche nombre`, "
             "`/confirmar` y `/cancelar`.")
    while True:
        try:
            texto = leer("Piloto > ").strip()
        except (EOFError, KeyboardInterrupt):
            escribir("Chat cerrado.")
            break
        if not texto:
            continue
        comando, _, argumento = texto.partition(" ")
        comando, argumento = comando.lower(), argumento.strip()
        if comando == "/salir":
            escribir("Chat cerrado.")
            break
        if comando == "/nuevo":
            config_chat = {"configurable": {"thread_id": uuid4().hex}}
            pendiente = False
            escribir("Conversación nueva con la selección actual. Escribe el síntoma inicial.")
            continue
        if comando == "/simulador":
            valor = argumento.lower()
            if valor not in permitidos:
                escribir("Elige `/simulador lmu`, `/simulador iracing` o `/simulador acc`.")
            else:
                if valor != seleccion["simulador"]:
                    seleccion["coche"] = None
                seleccion["simulador"] = valor
                pendiente = False
                escribir(f"Seleccionado {valor.upper()}. Escribe la consulta; confirmaremos el cambio si hay una sesión anterior.")
            continue
        if comando == "/coche":
            if not argumento:
                escribir("Indica el nombre después de `/coche`.")
            else:
                seleccion["coche"] = argumento
                pendiente = False
                escribir(f"Coche seleccionado: {argumento}. Escribe la consulta para continuar.")
            continue
        if comando == "/cancelar":
            estado = app.get_state(config_chat).values
            contexto = estado.get("contexto")
            if contexto:
                seleccion = dict(contexto)
            pendiente = False
            # La siguiente consulta con la selección original cancela el cambio en el grafo.
            escribir("Cambio cancelado. Se conserva el contexto anterior.")
            continue
        if comando == "/confirmar":
            if not pendiente:
                escribir("No hay un cambio pendiente de confirmar.")
                continue
            entrada = {"confirmar_contexto": True}
        elif texto.startswith("/"):
            escribir("Comando desconocido. Usa `/salir`, `/nuevo`, `/simulador`, `/coche`, `/confirmar` o `/cancelar`.")
            continue
        else:
            entrada = {"pregunta": texto, **seleccion}
        try:
            resultado = app.invoke(entrada, config=config_chat)
        except KeyboardInterrupt:
            escribir("Chat interrumpido.")
            break
        except Exception:
            pendiente = app.get_state(config_chat).values.get("requiere_confirmacion", False)
            escribir("No se ha podido completar la respuesta. Revisa la conexión o la cuota de Gemini y prueba más tarde. Puedes seguir usando el chat.")
            continue
        pendiente = resultado["requiere_confirmacion"]
        escribir(resultado["respuesta"] if ver_fuentes else resultado["respuesta_piloto"])
        if ver_fuentes and resultado["fuentes"]:
            escribir("**Fuentes para revisión**\n\n" + "\n".join(
                f"- [{f['id']}]({f['url']}) — {f['locator']}" for f in resultado["fuentes"]
            ))
        if pendiente:
            escribir("Usa `/confirmar` para procesar esa consulta en el contexto nuevo o `/cancelar` para conservar el anterior.")
    return config_chat


In [ ]:
config_chat = iniciar_chat(app, simulador="lmu", ver_fuentes=True)


### Cinco preguntas con respuestas reales

Escenarios ficticios del piloto, sin pruebas en pista. Las respuestas completas de D01 y D04 están guardadas en las secciones 4 y 5; D02, D03 y D05 aparecen debajo. Los informes registran modelo, fecha, versión de prompt, fragmentos y tokens.

| Caso | Simulador | Pregunta real | Dónde está la respuesta |
| --- | --- | --- | --- |
| D01 | lmu | En LMU, tengo sobreviraje al acelerar a la salida de una curva lenta. ¿Qué propone la guía? | Sección 4, ejemplo de voz |
| D02 | iracing | En iRacing bloqueo a menudo los neumáticos delanteros al frenar. ¿Qué reparto de frenada sugiere la chuleta? | Debajo de esta tabla |
| D03 | acc | En ACC el coche resulta incómodo e inestable al pasar por los pianos. ¿Qué ajustes indica la guía? | Debajo de esta tabla |
| D04 | lmu | Todavía no he cambiado nada. ¿Qué debería observar si pruebo solo tu primera propuesta? | Sección 5, primer seguimiento |
| D05 | lmu | ¿Qué presión exacta es óptima para mi Ferrari GT3 en la versión actual de LMU? | Debajo de esta tabla |

La muestra incluye revisiones diferentes del prompt. Las puntuaciones son revisiones manuales por caso, no una comparación global controlada.

#### D02 · iracing

**Piloto:** En iRacing bloqueo a menudo los neumáticos delanteros al frenar. ¿Qué reparto de frenada sugiere la chuleta?

**Respuesta real:**

Para poder ofrecerte una recomendación precisa y saber si el ajuste está disponible, necesito que confirmes primero qué coche estás utilizando en iRacing, ya que los reglajes y la respuesta a la frenada varían totalmente según el modelo.

Dicho esto, si sufres bloqueos frecuentes en el tren delantero, el objetivo técnico es lograr que los neumáticos delanteros bloqueen justo antes que los traseros para mantener la estabilidad [IR_CHRIS_HAYE_20]. Ante el problema que describes, la recomendación estándar es desplazar el reparto de frenada (brake bias) hacia atrás [IR_CHRIS_HAYE_02]. 

Como alternativa, también puedes reducir la presión de frenado general, aunque esto debe hacerse con precaución ya que afectará a tu capacidad de detener el coche [IR_CHRIS_HAYE_05].

¿Qué coche estás pilotando y qué reparto de frenada llevas configurado actualmente? Cuando me des el modelo, podremos verificar si el setup permite este ajuste.

**Localizadores verificados**

- [IR_CHRIS_HAYE_20](https://simracingcockpit.gg/improving-your-car-setup-in-iracing/) — Lámina / celda / Tips / Brake bias
- [IR_CHRIS_HAYE_02](https://simracingcockpit.gg/improving-your-car-setup-in-iracing/) — Lámina / celda / Braking / Front tyres often lock
- [IR_CHRIS_HAYE_05](https://simracingcockpit.gg/improving-your-car-setup-in-iracing/) — Lámina / celda / Braking / Excessive locking

**Resultado de revisión:** Reparto hacia atrás y criterio de bloqueo delantero coinciden con las fichas 02 y 20. La alternativa de presión general aparece en la 05 con cautela. Añade explicaciones sobre capacidad de detener el coche y pide el modelo para verificar ajustes, aunque el corpus no documenta coches concretos.

#### D03 · acc

**Piloto:** En ACC el coche resulta incómodo e inestable al pasar por los pianos. ¿Qué ajustes indica la guía?

**Respuesta real:**

Para poder asistirte, primero necesito que me confirmes con qué coche estás rodando en Assetto Corsa Competizione, ya que el comportamiento ante los pianos varía drásticamente según la plataforma y la configuración aerodinámica de cada modelo.

Mientras tanto, si sientes el coche incómodo al atacar pianos, esto apunta a un exceso de rigidez en la suspensión que impide que el coche absorba el impacto [ACC_DRIVER61_18]. Empezaría por comprobar el ajuste de los **muelles (wheel rate)**, reduciendo su valor para mejorar la capacidad de absorción del chasis.

Si después de ajustar el muelle el comportamiento persiste, prueba a **reducir la compresión y extensión rápida (fast bump y fast rebound)** [ACC_DRIVER61_18]. Esto permite que el amortiguador reaccione más libremente ante impactos súbitos como los de un piano. 

Ten en cuenta que estas recomendaciones provienen de una guía histórica (V1) y no podemos garantizar que todos estos ajustes estén disponibles o funcionen de la misma manera en la versión actual del juego o en tu coche específico. 

¿Qué coche estás utilizando para que podamos verificar qué ajustes permite tu configuración actual?

**Localizadores verificados**

- [ACC_DRIVER61_18](https://driver61.com/wp-content/uploads/2021/04/Driver61-Assetto-Corsa-Competizione-Setup-Guide-V1.pdf) — Página 1 / celda / Tyres / Uncomfortable taking kerbs

**Resultado de revisión:** Reducir wheel rate, fast bump y fast rebound coincide con la ficha 18 y declara antigüedad V1. El diagnóstico de rigidez y las explicaciones sobre absorción del impacto no están desarrollados en esa celda, que solo enumera ajustes.

#### D05 · lmu

**Piloto:** ¿Qué presión exacta es óptima para mi Ferrari GT3 en la versión actual de LMU?

**Respuesta real:**

No existe una cifra exacta de presión de neumáticos «óptima» que pueda proporcionarte para el Ferrari GT3 en Le Mans Ultimate. La documentación técnica disponible no especifica valores universales, rangos de trabajo ni presiones objetivo para este modelo o versión del simulador [LMU_USH_10], [LMU_USH_11].

Las presiones se ajustan en función de la gestión térmica del neumático en pista. En lugar de buscar una cifra predefinida, debemos trabajar con la temperatura de operación:

*   **Si tus neumáticos están demasiado calientes:** Aumentar la presión reduce la superficie de contacto del neumático, lo que disminuye la acumulación de calor [LMU_USH_10]. Ten precaución de no sobreinflarlos.
*   **Si tus neumáticos están demasiado fríos:** Disminuir la presión permite que el neumático flexione más bajo carga, distribuyendo el esfuerzo sobre una superficie mayor y aumentando su temperatura [LMU_USH_11].

Cualquier cambio debe realizarse de forma incremental y de uno en uno para poder evaluar su efecto real en tu ritmo y consistencia. 

¿Estás observando algún problema específico de temperatura o degradación que te preocupe en este momento? Si es así, indícame qué temperaturas estás viendo tras unas vueltas lanzadas y ajustaremos el procedimiento.

**Localizadores verificados**

- [LMU_USH_11](https://ultimatesetuphub.com/race-engineer/n3n4) — Ruta del árbol público / Tyre Temperature / Too Cold
- [LMU_USH_10](https://ultimatesetuphub.com/race-engineer/n3n3) — Ruta del árbol público / Tyre Temperature / Too Hot

**Revisión documental:** se excluyó una ficha contradictoria de las recomendaciones, conservando la recuperación original para auditoría.

**Resultado de revisión:** Declara que no dispone de presión exacta ni objetivos para modelo/versión. No inventa una cifra. Los mecanismos térmicos proceden de 10 y 11. La ficha contradictoria 08 se conserva en recuperados/excluidas y no se utiliza para elegir un ajuste del centro caliente. No valida físicamente un setup actual.

#### Revisión de los cinco casos

| Caso | Fidelidad (0–2) | Contexto (0–2) | Memoria | Citas |
| --- | --- | --- | --- | --- |
| D01 | 1 | 2 | No aplica a este caso aislado | Existentes; respaldo parcial |
| D02 | 1 | 1 | No aplica a este caso aislado | Existentes; respaldo parcial |
| D03 | 1 | 2 | No aplica a este caso aislado | Existentes; respaldo parcial |
| D04 | 1 | 2 | Recuerda propuesta pendiente | Existentes; respaldo parcial |
| D05 | 2 | 2 | No aplica a este caso aislado | Existentes; respaldo revisado para la pregunta |

Que las citas existan no valida todas las frases. Siguen pendientes correcciones de fidelidad. Se conservaron los intentos anteriores de D05 al revisar el conflicto de presiones; no se cuentan como preguntas distintas.


## 7. Bonus: Streamlit y presentación — pendiente

Chat web sobre el mismo núcleo y demo de 5–10 minutos.
